In [ ]:
# 1. We take the predictions and actual values ​​produced by the model into the CPU and convert them to numpy

preds_lstm = lstm_predictions.cpu().numpy()
preds_gru = gru_predictions.cpu().numpy()
gercek_degerler = y_test_tensor.numpy()

# 2. Using MinMaxScaler we convert these estimates back to the original Megawatt values ​​from the range -1, 1
# Note: we have defined the scaler object before, we inverse_transform it directly
gercek_mw = scaler.inverse_transform(gercek_degerler)
lstm_mw = scaler.inverse_transform(preds_lstm)
gru_mw = scaler.inverse_transform(preds_gru)

# 3. Creating a Comparison Table for the top 10 examples
print("-------------------------------------------------------------------------------")
print(f"{'Örnek':<8} | {'Gerçek Üretim (MW)':<20} | {'LSTM Tahmini (MW)':<18} | {'GRU Tahmini (MW)'}")
print("-------------------------------------------------------------------------------")

for i in range(10):
    print(f"{i+1:<8} | {gercek_mw[i][0]:<20.2f} | {lstm_mw[i][0]:<18.2f} | {gru_mw[i][0]:.2f}")

print("-------------------------------------------------------------------------------")

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

# 1. Device and Data Preparation
device = "cuda" if torch.cuda.is_available() else "cpu"
df = pd.read_csv('time_series_60min_singleindex.csv', parse_dates=['utc_timestamp'], index_col='utc_timestamp')
solar_data = df[['AT_solar_generation_actual']].dropna()

scaler = MinMaxScaler(feature_range=(-1, 1))
solar_data_scaled = scaler.fit_transform(solar_data.values)

def create_sequences(data, lookback):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:(i + lookback)])
        y.append(data[i + lookback])
    return np.array(X), np.array(y)

lookback = 24 
X, y = create_sequences(solar_data_scaled, lookback)

train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

X_train_tensor = torch.from_numpy(X_train).float()
y_train_tensor = torch.from_numpy(y_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_test_tensor = torch.from_numpy(y_test).float()

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
X_test_device = X_test_tensor.to(device)

criterion = nn.MSELoss()
epochs = 3  # Hızlı sonuç için 3 epoch tutuyoruz

# 2. LSTM Model Training and Prediction
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.layer_dim = layer_dim
        self.hidden_dim = hidden_dim

    def forward(self, x):
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

lstm_model = LSTMModel(1, 32, 2, 1).to(device)
optimizer_lstm = optim.Adam(lstm_model.parameters(), lr=0.001)

lstm_model.train()
for epoch in range(epochs):
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        loss = criterion(lstm_model(bx), by)
        optimizer_lstm.zero_grad()
        loss.backward()
        optimizer_lstm.step()

lstm_model.eval()
with torch.no_grad():
    lstm_preds = lstm_model(X_test_device).cpu().numpy()

# 3. GRU Model Training and Prediction
class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(GRUModel, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, layer_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.layer_dim = layer_dim
        self.hidden_dim = hidden_dim

    def forward(self, x):
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.gru(x, h0)
        out = self.fc(out[:, -1, :])
        return out

gru_model = GRUModel(1, 32, 2, 1).to(device)
optimizer_gru = optim.Adam(gru_model.parameters(), lr=0.001)

gru_model.train()
for epoch in range(epochs):
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        loss = criterion(gru_model(bx), by)
        optimizer_gru.zero_grad()
        loss.backward()
        optimizer_gru.step()

gru_model.eval()
with torch.no_grad():
    gru_preds = gru_model(X_test_device).cpu().numpy()

# 4. Converting to Megawatt (MW) and Creating a Table
gercek_mw = scaler.inverse_transform(y_test_tensor.numpy())
lstm_mw = scaler.inverse_transform(lstm_preds)
gru_mw = scaler.inverse_transform(gru_preds)

print("---------------------------------------------------------------")
print(f"{'Örnek':<8} | {'Gerçek Üretim (MW)':<20} | {'LSTM Tahmini (MW)':<18} | {'GRU Tahmini (MW)'}")
print("-------------------------------------------------------------------------------")

for i in range(10):
    print(f"{i+1:<8} | {gercek_mw[i][0]:<20.2f} | {lstm_mw[i][0]:<18.2f} | {gru_mw[i][0]:.2f}")

print("-------------------------------------------------------------------------------")